In [1]:
import sys
import os

# Add the src directory to sys.path
sys.path.append(os.path.abspath("../src"))

In [2]:
import random
import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset
from collections import defaultdict
from typing import List, Dict, Tuple, Any, Optional, Type, Union

from abstractions3d.primitives.shapes import Box3D
from abstractions3d.primitives.visualization import visualize_boxes_3d
from abstractions3d.dsl.core import Shape3D
from abstractions3d.dsl.nodes import Rect3D, Move3D, Union3D, SymTrans3D, SymRef3D
from abstractions3d.dsl.instantiation import instantiate_3d
from abstractions3d.data.blueprint import Table3D, Chair3D, Bench3D

In [3]:
class Autoencoder(nn.Module):
    """Simple autoencoder for parameter compression."""
    def __init__(self, input_dim: int, latent_dim: int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, input_dim)
        )

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        z = self.encoder(x)
        return z, self.decoder(z)

In [4]:
class AbstractionNode(Shape3D):
    """Node representing abstracted shape with latent parameters."""
    def __init__(self, type_list: List[Type[Shape3D]], latent: torch.Tensor, model: Optional[nn.Module] = None):
        super().__init__(children=[])
        self.type_list = type_list
        self.latent = latent
        self.model = model

    def expand(self) -> Shape3D:
        if self.model is None:
            param_list = self.latent.tolist()
        else:
            self.model.eval()
            with torch.no_grad():
                param_list = self.model.decoder(self.latent[None,:])[0].tolist()
        return instantiate_3d(self.type_list, param_list)

    def get_box3d_list(self) -> List[Box3D]:
        return self.expand().get_box3d_list()

    @classmethod
    def from_shape(cls, shape: Shape3D, models: Dict[str, nn.Module]) -> 'AbstractionNode':
        type_list = []
        numeric_params = []

        def collect(s: Shape3D) -> None:
            type_list.append(type(s))
            for p in s.param_tuple()[1]:
                if isinstance(p, (float, int)):
                    numeric_params.append(p)
            for c in getattr(s, 'children', []):
                collect(c)

        collect(shape)
        model = models.get(type(shape).__name__)
        latent = torch.tensor(numeric_params, dtype=torch.float32)
        return cls(type_list, latent, model)

In [5]:
def sample_uniform(low: float, high: float) -> float:
    return random.uniform(low, high)

def generate_dataset(num_samples: int = 10) -> List[Shape3D]:
    dataset = []
    for _ in range(num_samples):
        # Tables
        table = Table3D(
            top_length=sample_uniform(3.0, 6.0),
            top_depth=sample_uniform(1.5, 3.0),
            top_thickness=sample_uniform(0.1, 0.3),
            leg_length=sample_uniform(0.2, 0.4),
            leg_depth=sample_uniform(0.2, 0.4),
            leg_height=sample_uniform(1.0, 2.0)
        )
        dataset.append(table)

        # Chairs
        chair = Chair3D(
            seat_length=sample_uniform(1.0, 2.0),
            seat_depth=sample_uniform(1.0, 2.0),
            seat_thickness=sample_uniform(0.2, 0.5),
            leg_length=sample_uniform(0.1, 0.3),
            leg_depth=sample_uniform(0.1, 0.3),
            leg_height=sample_uniform(0.5, 1.0),
            backrest_height=sample_uniform(1.0, 2.0),
            backrest_thickness=sample_uniform(0.1, 0.3)
        )
        dataset.append(chair)

        # Benches
        bench = Bench3D(
            seat_length=sample_uniform(2.0, 5.0),
            seat_depth=sample_uniform(0.5, 1.0),
            seat_thickness=sample_uniform(0.2, 0.5),
            leg_length=sample_uniform(0.1, 0.3),
            leg_depth=sample_uniform(0.1, 0.3),
            leg_height=sample_uniform(0.5, 1.0),
            backrest_height=sample_uniform(0.5, 1.5),
            backrest_thickness=sample_uniform(0.1, 0.3)
        )
        dataset.append(bench)
    return dataset

In [6]:
def add_dicts(d1: Dict[str, List], d2: Dict[str, List]) -> Dict[str, List]:
    for k in d2:
        d1[k].extend(d2[k])
    return d1

def get_singletons(shapes: Union[Shape3D, List[Shape3D]]) -> Dict[str, List[Tuple[float,...]]]:
    singletons = defaultdict(list)
    if isinstance(shapes, list):
        for s in shapes:
            singletons = add_dicts(singletons, get_singletons(s))
        return singletons
    t, params = shapes.param_tuple()
    floats = tuple(p for p in params if isinstance(p, (float,int)))
    if floats:
        singletons[t.__name__].append(floats)
    for c in getattr(shapes, 'children', []):
        singletons = add_dicts(singletons, get_singletons(c))
    return singletons

def get_pairs(shapes: Union[Shape3D, List[Shape3D]]) -> Dict[str, List[Tuple[float,...]]]:
    pairs = defaultdict(list)
    if isinstance(shapes, list):
        for s in shapes:
            pairs = add_dicts(pairs, get_pairs(s))
        return pairs
    if isinstance(shapes, Union3D):
        t, (c1, c2) = shapes.param_tuple()
        t1, p1 = c1.param_tuple()
        t2, p2 = c2.param_tuple()
        key = f"{t.__name__}({t1.__name__},{t2.__name__})"
        pairs[key].append(tuple([p for p in p1 if isinstance(p,(float,int))] +
                          [p for p in p2 if isinstance(p,(float,int))]))
        for c in shapes.children:
            pairs = add_dicts(pairs, get_pairs(c))
    else:
        for c in getattr(shapes, 'children', []):
            pairs = add_dicts(pairs, get_pairs(c))
    return pairs

In [7]:
def find_abstractions(structures: Dict[str,List[Tuple[float,...]]],
                     retrain_iterations: int = 2,
                     error_threshold: float = 0.01) -> Dict[str, nn.Module]:
    models = {}
    for name, parameters in structures.items():
        if not parameters or not isinstance(parameters[0], (tuple,list)):
            continue
        float_params = [list(p) for p in parameters if any(isinstance(x,(float,int)) for x in p)]
        if not float_params: continue
        num_float_parameters = len(float_params[0])
        if num_float_parameters < 2: continue

        train_tensor = torch.tensor(float_params,dtype=torch.float32)
        train_dl = DataLoader(TensorDataset(train_tensor), batch_size=64, shuffle=True)
        model = Autoencoder(input_dim=num_float_parameters, latent_dim=max(1,num_float_parameters-1))
        optimizer = AdamW(model.parameters(), lr=1e-3)
        loss_fn = lambda pred,target: torch.mean(torch.max(torch.abs(pred-target),dim=-1)[0])

        model.train()
        for _ in range(retrain_iterations):
            for batch in train_dl:
                x = batch[0]
                optimizer.zero_grad()
                _, x_rec = model(x)
                loss = loss_fn(x_rec,x)
                loss.backward()
                optimizer.step()
        models[name] = model
        print(f"Trained model for {name} with {num_float_parameters} parameters")
    return models

In [8]:
def integrate_abstractions(shape: Shape3D, models: Dict[str, nn.Module],
                           error_threshold: float = 0.01,
                           stats: Optional[Dict[str,int]] = None) -> Shape3D:
    if stats is None:
        stats = {'replaced':0}

    # Composite shapes
    if isinstance(shape, Union3D):
        new_child1 = integrate_abstractions(shape.children[0], models, error_threshold, stats)
        new_child2 = integrate_abstractions(shape.children[1], models, error_threshold, stats)

        type_str = f"Union3D({type(new_child1).__name__},{type(new_child2).__name__})"
        params1 = [p for p in new_child1.param_tuple()[1] if isinstance(p,(float,int))]
        params2 = [p for p in new_child2.param_tuple()[1] if isinstance(p,(float,int))]
        float_params = params1 + params2

        if type_str in models and float_params:
            model = models[type_str]
            input_tensor = torch.tensor(float_params)[None,:]
            with torch.no_grad():
                _, recon = model(input_tensor)
                error = torch.max(torch.abs(recon - input_tensor)).item()
            if error < error_threshold:
                type_list = [Union3D,type(new_child1),type(new_child2)]
                latent = torch.tensor(float_params,dtype=torch.float32)
                stats['replaced'] += 1
                return AbstractionNode(type_list, latent, model)
        return Union3D(new_child1,new_child2)

    # Singleton shapes
    type_, params = shape.param_tuple()
    float_params = [p for p in params if isinstance(p,(float,int))]
    type_str = type_.__name__
    if type_str in models and float_params:
        model = models[type_str]
        input_tensor = torch.tensor(float_params)[None,:]
        with torch.no_grad():
            _, recon = model(input_tensor)
            error = torch.max(torch.abs(recon - input_tensor)).item()
        if error < error_threshold:
            type_list = [type_]
            latent = torch.tensor(float_params,dtype=torch.float32)
            stats['replaced'] += 1
            return AbstractionNode(type_list, latent, model)

    new_children = [integrate_abstractions(c, models, error_threshold, stats) 
                    for c in getattr(shape,'children',[])]
    shape.children = new_children
    return shape

In [16]:
# Generate dataset
dataset = generate_dataset(num_samples=200)
print(f"Generated {len(dataset)} shapes")

Generated 600 shapes


In [17]:
# Visualize random shape
shape = random.choice(dataset)
visualize_boxes_3d(shape.get_box3d_list())

Output()

In [18]:
# Extract structures
singletons = get_singletons(dataset)
pairs = get_pairs(dataset)
structures = add_dicts(singletons, pairs)
print("Extracted structure types:", structures.keys())

Extracted structure types: dict_keys(['Move3D', 'Rect3D', 'Union3D(Move3D,SymRef3D)', 'Union3D(Move3D,Union3D)', 'Union3D(SymRef3D,Move3D)'])


In [23]:
# Cell 5 — Find abstractions
def find_abstractions(structures: Dict[str,List[Tuple[float,...]]],
                      retrain_iterations: int = 10,
                      error_threshold: float = 0.05) -> Tuple[Dict[str, nn.Module], Dict[str,List[float]]]:
    models = {}
    losses = {}
    for name, params in structures.items():
        if not params: continue
        float_params = [list(p) for p in params if any(isinstance(x,(float,int)) for x in p)]
        if not float_params: continue
        num_float = len(float_params[0])
        if num_float < 2: continue

        train_tensor = torch.tensor(float_params,dtype=torch.float32)
        train_dl = DataLoader(TensorDataset(train_tensor), batch_size=32, shuffle=True)
        model = Autoencoder(input_dim=num_float, latent_dim=max(1,num_float-1))
        optimizer = AdamW(model.parameters(), lr=1e-3)
        loss_fn = lambda pred,target: torch.mean(torch.max(torch.abs(pred-target),dim=-1)[0])

        model.train()
        losses[name] = []
        for _ in range(retrain_iterations):
            batch_losses = []
            for batch in train_dl:
                x = batch[0]
                optimizer.zero_grad()
                _, x_rec = model(x)
                loss = loss_fn(x_rec,x)
                loss.backward()
                optimizer.step()
                batch_losses.append(loss.item())
            losses[name].append(sum(batch_losses)/len(batch_losses))
        models[name] = model
        print(f"Trained {name}, final loss: {losses[name][-1]:.4f}")
    return models, losses

In [24]:
# Cell 6 — Integrate abstractions
def integrate_abstractions(shape: Shape3D, models: Dict[str, nn.Module],
                           error_threshold: float = 0.05,
                           stats: Optional[Dict[str,int]] = None) -> Shape3D:
    if stats is None: stats = {'replaced':0}

    # Composite
    if isinstance(shape,Union3D):
        new_c1 = integrate_abstractions(shape.children[0],models,error_threshold,stats)
        new_c2 = integrate_abstractions(shape.children[1],models,error_threshold,stats)
        type_str = f"Union3D({type(new_c1).__name__},{type(new_c2).__name__})"
        float_params = [p for p in new_c1.param_tuple()[1] if isinstance(p,(float,int))] + \
                       [p for p in new_c2.param_tuple()[1] if isinstance(p,(float,int))]
        if type_str in models and float_params:
            model = models[type_str]
            input_tensor = torch.tensor(float_params)[None,:]
            with torch.no_grad():
                _, recon = model(input_tensor)
                error = torch.max(torch.abs(recon - input_tensor)).item()
            if error < error_threshold:
                stats['replaced'] += 1
                return AbstractionNode([Union3D,type(new_c1),type(new_c2)],
                                       torch.tensor(float_params,dtype=torch.float32),
                                       model)
        return Union3D(new_c1,new_c2)

    # Singleton
    type_, params = shape.param_tuple()
    float_params = [p for p in params if isinstance(p,(float,int))]
    type_str = type_.__name__
    if type_str in models and float_params:
        model = models[type_str]
        input_tensor = torch.tensor(float_params)[None,:]
        with torch.no_grad():
            _, recon = model(input_tensor)
            error = torch.max(torch.abs(recon - input_tensor)).item()
        if error < error_threshold:
            stats['replaced'] += 1
            return AbstractionNode([type_], torch.tensor(float_params,dtype=torch.float32), model)

    new_children = [integrate_abstractions(c,models,error_threshold,stats) 
                    for c in getattr(shape,'children',[])]
    shape.children = new_children
    return shape

In [25]:
# Cell 7 — Full pipeline
dataset = generate_dataset(num_samples=50)
print(f"Generated {len(dataset)} shapes")

shape = random.choice(dataset)
visualize_boxes_3d(shape.get_box3d_list())

singletons = get_singletons(dataset)
pairs = get_pairs(dataset)
structures = add_dicts(singletons,pairs)
print("Extracted structures:", structures.keys())

models, losses = find_abstractions(structures)

abstracted_dataset = []
stats_list = []
for shape in dataset:
    stats = {'replaced':0}
    abstracted = integrate_abstractions(shape,models,error_threshold=0.05,stats=stats)
    abstracted_dataset.append(abstracted)
    stats_list.append(stats)

total_replaced = sum(s['replaced'] for s in stats_list)
print(f"Total nodes replaced: {total_replaced}, avg per shape: {total_replaced/len(dataset):.2f}")

Generated 150 shapes


Output()

Extracted structures: dict_keys(['Move3D', 'Rect3D', 'Union3D(Move3D,SymRef3D)', 'Union3D(Move3D,Union3D)', 'Union3D(SymRef3D,Move3D)'])
Trained Move3D, final loss: 0.3833
Trained Rect3D, final loss: 0.5854
Trained Union3D(Move3D,SymRef3D), final loss: 1.4332
Trained Union3D(Move3D,Union3D), final loss: 0.1673
Trained Union3D(SymRef3D,Move3D), final loss: 1.5666
Total nodes replaced: 39, avg per shape: 0.26


In [26]:
# Cell 8 — Visualize original vs abstracted
original = dataset[0]
abstracted = abstracted_dataset[0]

print("Original shape")
visualize_boxes_3d(original.get_box3d_list())

print("Abstracted shape")
visualize_boxes_3d(abstracted.get_box3d_list())

Original shape


Output()

Abstracted shape


Output()

In [28]:
print(abstracted)

Union3D(
    Move3D(
        Rect3D(
            5.332,
            0.105,
            2.425
        ),
        0.000,
        1.905,
        0.000
    ),
    SymRef3D(
        SymRef3D(
            Move3D(
                Rect3D(
                    0.395,
                    1.853,
                    0.389
                ),
                2.419,
                0.927,
                0.968
            ),
            x
        ),
        z
    )
)


In [29]:
for i, stats in enumerate(stats_list):
    if stats['replaced'] > 0:
        print(f"Shape {i} had {stats['replaced']} nodes replaced by abstractions")

Shape 1 had 1 nodes replaced by abstractions
Shape 5 had 1 nodes replaced by abstractions
Shape 8 had 1 nodes replaced by abstractions
Shape 11 had 1 nodes replaced by abstractions
Shape 14 had 1 nodes replaced by abstractions
Shape 19 had 1 nodes replaced by abstractions
Shape 22 had 1 nodes replaced by abstractions
Shape 23 had 1 nodes replaced by abstractions
Shape 26 had 1 nodes replaced by abstractions
Shape 28 had 1 nodes replaced by abstractions
Shape 40 had 1 nodes replaced by abstractions
Shape 44 had 1 nodes replaced by abstractions
Shape 53 had 1 nodes replaced by abstractions
Shape 58 had 1 nodes replaced by abstractions
Shape 59 had 1 nodes replaced by abstractions
Shape 65 had 1 nodes replaced by abstractions
Shape 67 had 1 nodes replaced by abstractions
Shape 70 had 1 nodes replaced by abstractions
Shape 71 had 1 nodes replaced by abstractions
Shape 74 had 1 nodes replaced by abstractions
Shape 76 had 1 nodes replaced by abstractions
Shape 77 had 1 nodes replaced by abst

In [32]:
index_to_check = 1  # choose any shape index
original = dataset[index_to_check]
abstracted = abstracted_dataset[index_to_check]

print(f"Shape {index_to_check} — Original vs Abstracted")

# Original
print("Original shape boxes:")
print(original)
#visualize_boxes_3d(original.get_box3d_list())

# Abstracted
print("Abstracted shape boxes:")
print(abstracted)
#visualize_boxes_3d(abstracted.get_box3d_list())

Shape 1 — Original vs Abstracted
Original shape boxes:
Union3D(
    Move3D(
        Rect3D(
            1.782,
            0.339,
            1.624
        ),
        0.000,
        1.110,
        0.000
    ),
    Union3D(
        SymRef3D(
            SymRef3D(
                Move3D(
                    Shape3D,
                    0.767,
                    0.470,
                    0.670
                ),
                x
            ),
            z
        ),
        Move3D(
            Rect3D(
                1.782,
                1.466,
                0.271
            ),
            0.000,
            2.012,
            -0.676
        )
    )
)
Abstracted shape boxes:
Union3D(
    Move3D(
        Rect3D(
            1.782,
            0.339,
            1.624
        ),
        0.000,
        1.110,
        0.000
    ),
    Union3D(
        SymRef3D(
            SymRef3D(
                Move3D(
                    Shape3D,
                    0.767,
                    0

In [33]:
def has_abstraction_node(shape: Shape3D) -> bool:
    """Check recursively if a shape contains any AbstractionNode."""
    if isinstance(shape, AbstractionNode):
        return True
    for c in getattr(shape, 'children', []):
        if has_abstraction_node(c):
            return True
    return False

# Print dataset indices that contain AbstractionNode
for i, abs_shape in enumerate(abstracted_dataset):
    if has_abstraction_node(abs_shape):
        print(f"Shape at index {i} contains an AbstractionNode")

Shape at index 1 contains an AbstractionNode
Shape at index 5 contains an AbstractionNode
Shape at index 8 contains an AbstractionNode
Shape at index 11 contains an AbstractionNode
Shape at index 14 contains an AbstractionNode
Shape at index 19 contains an AbstractionNode
Shape at index 22 contains an AbstractionNode
Shape at index 23 contains an AbstractionNode
Shape at index 26 contains an AbstractionNode
Shape at index 28 contains an AbstractionNode
Shape at index 40 contains an AbstractionNode
Shape at index 44 contains an AbstractionNode
Shape at index 53 contains an AbstractionNode
Shape at index 58 contains an AbstractionNode
Shape at index 59 contains an AbstractionNode
Shape at index 65 contains an AbstractionNode
Shape at index 67 contains an AbstractionNode
Shape at index 70 contains an AbstractionNode
Shape at index 71 contains an AbstractionNode
Shape at index 74 contains an AbstractionNode
Shape at index 76 contains an AbstractionNode
Shape at index 77 contains an Abstrac

In [19]:
# Train abstraction models
models = find_abstractions(structures)

Trained model for Move3D with 3 parameters
Trained model for Rect3D with 3 parameters
Trained model for Union3D(Move3D,SymRef3D) with 3 parameters
Trained model for Union3D(Move3D,Union3D) with 3 parameters
Trained model for Union3D(SymRef3D,Move3D) with 3 parameters


In [20]:
# Integrate abstractions
abstracted_dataset = []
stats_list = []
for shape in dataset:
    stats = {'replaced':0}
    abstracted_shape = integrate_abstractions(shape, models, error_threshold=0.01, stats=stats)
    abstracted_dataset.append(abstracted_shape)
    stats_list.append(stats)

In [21]:
# Summary
total_shapes = len(dataset)
total_replaced = sum(s['replaced'] for s in stats_list)
print(f"Total shapes: {total_shapes}")
print(f"Total abstracted nodes: {total_replaced}")
print(f"Average per shape: {total_replaced/total_shapes:.2f}")

Total shapes: 600
Total abstracted nodes: 0
Average per shape: 0.00


In [22]:
# Show first 5 shapes stats
for i,s in enumerate(stats_list[:5]):
    print(f"Shape {i} replaced nodes: {s['replaced']}")

Shape 0 replaced nodes: 0
Shape 1 replaced nodes: 0
Shape 2 replaced nodes: 0
Shape 3 replaced nodes: 0
Shape 4 replaced nodes: 0
